In [ ]:
import os
import shutil
import subprocess
import sys
import time

# --- 1. INSTALL DEPENDENCIES ---
print(">>> Installing required libraries...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "unidecode", "-q"])

# --- 2. DYNAMIC PATH DISCOVERY ---
print("\n=== STEP 1: Locating Project Directories ===")
SRC_DIR, DATA_DIR = None, None

for root, _, files in os.walk("/kaggle/input"):
    if "prepare.py" in files and not SRC_DIR:
        SRC_DIR = root
        print(f"[FOUND] Source Code: {SRC_DIR}")
    if "train_source1.tsv" in files and not DATA_DIR:
        DATA_DIR = os.path.dirname(root) if os.path.basename(root) == "train" else root
        print(f"[FOUND] Dataset:     {DATA_DIR}")

if not SRC_DIR or not DATA_DIR:
    raise FileNotFoundError("Could not find the source code or dataset folders.")

# --- 3. WIPE & PREPARE WRITABLE WORKSPACE ---
print("\n=== STEP 2: Setting Up Clean Workspace ===")
WORK_BASE = "/kaggle/working"
SRC_WORK = os.path.join(WORK_BASE, "src")
WORK_DIR = os.path.join(WORK_BASE, "work")
OUT_DIR = os.path.join(WORK_BASE, "output")

# Force-clean old corrupted runs
for d in [SRC_WORK, WORK_DIR, OUT_DIR]:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

shutil.copytree(SRC_DIR, SRC_WORK, dirs_exist_ok=True)

# --- 4. AUTO-APPLY ALL PATCHES & F0.5 TUNING ---
print("\n=== STEP 3: Applying Fixes & Score Optimizations ===")

def patch_file(filename, replacements):
    path = os.path.join(SRC_WORK, filename)
    if os.path.exists(path):
        with open(path, "r") as f:
            content = f.read()
        for old, new in replacements:
            content = content.replace(old, new)
        with open(path, "w") as f:
            f.write(content)
        print(f"[PATCHED] {filename}")

# Fix 1: The 'f_nsp1' bug and the Polars 'is_in' shape mismatch in features.py
patch_file("features.py", [
    ("f_nsp1", "nsp"),
    ("is_in(need.implode())", "is_in(need)")
])

# Fix 2: Inject F0.5 Precision Booster into predict.py
# We force the model to drop low-confidence matches to optimize the F0.5 penalty
pred_target = "sel = dec.select(P.select('t_rid', 's1_rid'), P['p'].to_numpy())"
pred_replacement = (
    "sel = dec.select(P.select('t_rid', 's1_rid'), P['p'].to_numpy())\n"
    "    # BOOSTER: Force strict precision for F0.5 score\n"
    "    if 'p' in sel.columns:\n"
    "        sel = sel.filter(pl.col('p') >= 0.76)\n"
)
patch_file("predict.py", [(pred_target, pred_replacement)])

# --- 5. EXECUTE OPTIMIZED PIPELINE ---
print("\n=== STEP 4: Executing Streamlined ML Pipeline ===")

# Force max multithreading on Kaggle's 4 vCPUs
env = os.environ.copy()
env["ER_DATA_DIR"] = DATA_DIR
env["ER_WORK_DIR"] = WORK_DIR
env["ER_OUT_DIR"] = OUT_DIR
env["ER_WORKERS"] = "4"
env["OMP_NUM_THREADS"] = "4"
env["POLARS_MAX_THREADS"] = "4"
env["MKL_NUM_THREADS"] = "4"

# We drop trees from 1500 to 800 to significantly speed up training with virtually zero accuracy loss
stages = [
    ["python", "prepare.py", "train", "test"],
    ["python", "run_blocking.py", "train", "10", "300"],
    ["python", "run_blocking.py", "test", "10", "300"],
    ["python", "train.py", "all", "0.25", "800"], 
    ["python", "predict.py", "all"]
]

for cmd in stages:
    step_str = " ".join(cmd)
    print(f"\n>>> Running: {step_str}")
    stage_t0 = time.time()
    
    res = subprocess.run(cmd, cwd=SRC_WORK, env=env)
    
    elapsed = (time.time() - stage_t0) / 60
    if res.returncode != 0:
        raise RuntimeError(f"Pipeline crashed at '{step_str}' with exit code {res.returncode}")
    print(f">>> Finished {step_str} in {elapsed:.2f} minutes.")

print("\n" + "=" * 60)
print("SUCCESS! Pipeline execution complete.")
print(f"Your optimized submission files are ready in: {OUT_DIR}")
print("=" * 60)

>>> Installing required libraries...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 5.3 MB/s eta 0:00:00

=== STEP 1: Locating Project Directories ===
[FOUND] Source Code: /kaggle/input/datasets/prattttttttttttt/amazon-ml-code/amazon-ml-challenge-main/src
[FOUND] Dataset:     /kaggle/input/datasets/prattttttttttttt/amazon-dataset/dataset

=== STEP 2: Setting Up Clean Workspace ===

=== STEP 3: Applying Fixes & Score Optimizations ===
[PATCHED] features.py
[PATCHED] predict.py

=== STEP 4: Executing Streamlined ML Pipeline ===

>>> Running: python prepare.py train test
translit table: 1362 entries (40s)
  train S1 rows 0+ (47s)
  train S1 rows 1,000,000+ (92s)
  train S1 rows 2,000,000+ (101s)
  train S2 rows 0+ (152s)
  train S2 rows 1,000,000+ (205s)
  train S2 rows 2,000,000+ (255s)
  train S3 rows 0+ (413s)
  train S3 rows 1,000,000+ (462s)
  train S3 rows 2,000,000+ (511s)
  train S3 rows 3,000,000+ (561s)
  train S3 rows 4,000,000+ (611s)
  train S3 rows 5,000,000+ (62